# **Лабораторная работа №2**

---


**Задание 1. Создание тензоров + приведение форм**

* Создайте тензор T размером (4, 6):
первая строка — все единицы,
остальные строки — случайные torch.rand.

* Создайте вектор v = torch.arange(24) и превратите его в матрицу (4, 6).

* Сложите T + v (проверьте broadcasting), выведите результат и форму.






In [ ]:
import torch

# создаем тензор T (4, 6)
T = torch.rand(4, 6)
T[0, :] = 1.0  # первая строка - единицы

# создаем вектор v и меняем форму
v = torch.arange(24).view(4, 6)

# складываем (Broadcasting не нужен, так как формы (4,6) и (4,6) совпадают)
res = T + v

print(f"Результат:\n{res}")
print(f"Форма результата: {res.shape}")

Результат:
tensor([[ 1.0000,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000],
        [ 6.2696,  7.1932,  8.7230,  9.8108, 10.1208, 11.5125],
        [12.1470, 13.1738, 14.6001, 15.1712, 16.5326, 17.9790],
        [18.2842, 19.3901, 20.1949, 21.7459, 22.0890, 23.0349]])
Форма результата: torch.Size([4, 6])




---



**Задание 2. Индексация и маски**

Дан A = torch.randn(5, 5).

*   Замените все элементы меньше 0 на 0 (без циклов).
*   Замените диагональ на 1 (можно через torch.diag/индексацию).
*   Найдите количество элементов, которые были заменены на 0.





In [ ]:
A = torch.randn(5, 5)

# считаем количество отрицательных элементов до замены
replaced_count = (A < 0).sum().item()

# заменяем отрицательные на 0
A[A < 0] = 0

# заменяем диагональ на 1
A.diagonal().fill_(1)

print(f"Матрица A:\n{A}")
print(f"Количество замененных элементов: {replaced_count}")

Матрица A:
tensor([[1.0000, 1.2231, 0.7401, 1.6501, 0.0000],
        [0.0000, 1.0000, 1.0743, 0.6146, 0.0000],
        [0.0627, 2.0367, 1.0000, 0.0000, 0.0000],
        [0.8965, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.2781, 0.0000, 0.0000, 0.0000, 1.0000]])
Количество замененных элементов: 14




---



**Задание 3. Matmul + проверка размерностей**


*   Создайте X размером (16, 7), W размером (7, 3), b размером (3,).
*   Посчитайте Z = X @ W + b и получите форму (16, 3).
*   Затем получите Z_alt = W.T @ X.T и приведите к форме (16, 3).
*   Проверьте, что Z и Z_alt совпадают (или объясните, почему не совпадают).


In [ ]:
X = torch.randn(16, 7)
W = torch.randn(7, 3)
b = torch.randn(3)

# классический forward
Z = X @ W + b

# Z_alt через транспонирование
# (W.T @ X.T) имеет форму (3, 16), поэтому в конце снова транспонируем в (16, 3)
Z_alt = (W.T @ X.T).T + b

# проверка совпадения (с учетом точности float)
matches = torch.allclose(Z, Z_alt)
print(f"Форма Z: {Z.shape}")
print(f"Z и Z_alt совпадают: {matches}")

Форма Z: torch.Size([16, 3])
Z и Z_alt совпадают: True




---



**Задание 4. Полносвязная сеть**

Соберите сеть:

image.png

где:
*   X: (64, 12)
*   W1: (12, 8)
*   b1: (8,)
*   W2: (8, 2)
*   b2: (2,)

Реализуйте forward.

Проверьте формы на каждом шаге.

Посчитайте loss как MSE между Z3 и целевым Y размера (64, 2).


In [ ]:
import torch
import torch.nn.functional as F

# инициализация данных и параметров
X = torch.randn(64, 12)   # входные данные
W1 = torch.randn(12, 8)  # веса первого слоя
b1 = torch.randn(8)      # смещение первого слоя
W2 = torch.randn(8, 2)   # веса второго слоя
b2 = torch.randn(2)      # смещение второго слоя
Y = torch.randn(64, 2)   # целевые значения (Ground Truth)

# реализация Forward pass с проверкой форм
# шаг 1: линейное преобразование 1
Z1 = X @ W1 + b1
print(f"Shape Z1: {Z1.shape}") # ожидаем (64, 8)

# шаг 2: функция активации ReLU
Z2 = torch.relu(Z1)
print(f"Shape Z2: {Z2.shape}") # ожидаем (64, 8)

# шаг 3: линейное преобразование 2 (выходной слой)
Z3 = Z2 @ W2 + b2
print(f"Shape Z3: {Z3.shape}") # Ожидаем (64, 2)

# расчет Loss (MSE)
loss = F.mse_loss(Z3, Y)

print(f"\nFinal Loss: {loss.item():.4f}")

Shape Z1: torch.Size([64, 8])
Shape Z2: torch.Size([64, 8])
Shape Z3: torch.Size([64, 2])

Final Loss: 46.3954




---



**Задание 5. Autograd: градиенты и проверка вручную**

*   Создайте скаляр x и y с requires_grad=True.
*   Определите:

image.png
*   Найдите df/dx и df/dy через .backward().
*   Проверьте df/dx аналитически хотя бы для части выражения (например для 5(х^2)y).




In [ ]:
import torch

# создаем скаляры с возможностью вычисления градиента
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# определяем функцию
f = 5 * (x**2) * y + torch.sin(x) + y**3

# вычисляем градиенты
f.backward()

print(f"df/dx (PyTorch): {x.grad.item():.4f}")
print(f"df/dy (PyTorch): {y.grad.item():.4f}")

df/dx (PyTorch): 59.5839
df/dy (PyTorch): 47.0000




---



Аналитическая проверка:

Проверим производную $\frac{\partial f}{\partial x}$ вручную
1. Производная первого слагаемого $5x^2y$ по $x$ -> $10xy$.
2. Производная $\sin(x)$ по $x$ -> $\cos(x)$.
3. Производная $y^3$ по $x$ -> $0$.

Итоговая формула $\frac{\partial f}{\partial x} = 10xy + \cos(x)$.

Подставим значения ($x=2, y=3$):

$$\frac{\partial f}{\partial x} = 10 \cdot 2 \cdot 3 + \cos(2) = 60 + (-0.4161) = 59.5839$$



---



**Задание 6**

.detach() — заморозка части графа
* Создайте a и b с requires_grad=True.
* Посчитайте: $u = a \cdot b, \quad v = u.detach() \cdot a, \quad p = v.sum()$
* Вызовите p.backward().

Ответьте:
* будет ли градиент по b?
* будет ли градиент по a?
* почему?


In [ ]:
a = torch.randn(3, 3, requires_grad=True)
b = torch.randn(3, 3, requires_grad=True)

u = a * b
v = u.detach() * a  # u отрывается от графа
p = v.sum()

p.backward()

print(f"Градиент по a:\n{a.grad}")
print(f"Градиент по b:\n{b.grad}")

Градиент по a:
tensor([[-0.0202, -0.5186,  0.4358],
        [-0.3143,  5.6985,  0.1317],
        [ 1.0539, -1.5205,  0.4712]])
Градиент по b:
None


Ответы на вопросы:

1. Будет ли градиент по b?

Нет (выведет None).

2. Будет ли градиент по a?

Да, градиент будет вычислен.

3. Почему?

Метод .detach() создает "копию" тензора u, которая не участвует в вычислении градиентов. Поскольку переменная b участвовала в создании p только через u, связь между p и b в вычислительном графе обрывается. Переменная a при этом всё еще напрямую участвует в вычислении v = u.detach() * a, поэтому путь для градиента к a остается открытым.



---



**Задание 7. CPU vs GPU**

* Создайте X = torch.randn(2000, 2000).

* Измерьте время вычисления X @ X на CPU.

* Если доступна CUDA — перенесите на GPU и измерьте время там.

* Сделайте вывод: где быстрее и почему (2–3 предложения).


In [ ]:
import torch
import time

# подготовка данных на CPU
X = torch.randn(2000, 2000)

# измерение времени на CPU
start_cpu = time.time()
res_cpu = X @ X
end_cpu = time.time()
cpu_duration = end_cpu - start_cpu
print(f"Время выполнения на CPU: {cpu_duration:.4f} сек")

# измерение времени на GPU (если доступна CUDA)
if torch.cuda.is_available():
    device = torch.device("cuda")
    X_gpu = X.to(device)

    # прогревочный запуск (warm-up), чтобы исключить время инициализации CUDA
    _ = X_gpu @ X_gpu

    # синхронизируем выполнение, так как операции на GPU асинхронны
    torch.cuda.synchronize()

    start_gpu = time.time()
    res_gpu = X_gpu @ X_gpu
    torch.cuda.synchronize()
    end_gpu = time.time()

    gpu_duration = end_gpu - start_gpu
    print(f"Время выполнения на GPU: {gpu_duration:.4f} сек")
    print(f"Ускорение: {cpu_duration / gpu_duration:.1f}x")
else:
    print("CUDA недоступна")

Время выполнения на CPU: 0.1571 сек
Время выполнения на GPU: 0.0047 сек
Ускорение: 33.2x


**Вывод**

Вычисления на GPU проходят значительно быстрее, так как архитектура графического процессора оптимизирована для параллельной обработки огромных массивов данных. В то время как CPU имеет небольшое количество мощных ядер для последовательных задач, GPU содержит тысячи мелких ядер, которые могут одновременно выполнять операцию умножения для разных ячеек матрицы.



---

